# Профилирование и качество данных Airbnb

**ЛР № 3. Исправленная версия**

In [ ]:
import duckdb
import os

# Настройки подключения из .env (без дублирования)
s3_key = os.getenv('S3_ACCESS_KEY', 's3admin')
s3_secret = os.getenv('S3_SECRET_KEY', 's3admin123')
s3_endpoint = os.getenv('S3_ENDPOINT', 'localhost:8333')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET seaweed (TYPE s3, KEY_ID '{s3_key}', SECRET '{s3_secret}', ENDPOINT '{s3_endpoint}', URL_STYLE 'path', USE_SSL false);")

raw_path = 's3://raw/nyc-airbnb-listings/ingested_on=2026-11-03/listings.csv'
staging_path = 's3://staging/nyc-airbnb-listings/source_ingested_on=2026-11-03/staging.parquet'
quarantine_path = 's3://staging/nyc-airbnb-listings/source_ingested_on=2026-11-03/quarantine.parquet'

## Шаг 2. Доработанный профиль данных

In [ ]:
print("1. Типы полей:")
for row in con.execute(f"DESCRIBE SELECT id, price, neighbourhood_group, availability_365 FROM read_csv_auto('{raw_path}')").fetchall():
    print(row)

print("\n2. Пропуски в price:")
print(con.execute(f"SELECT COUNT(*) as total, SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) as nulls FROM read_csv_auto('{raw_path}')").fetchall())

print("\n3. Дубликаты по id:")
print(con.execute(f"SELECT id, COUNT(*) FROM read_csv_auto('{raw_path}') GROUP BY id HAVING COUNT(*) > 1 LIMIT 3").fetchall())

print("\n4. Распределение по районам:")
for row in con.execute(f"SELECT neighbourhood_group, COUNT(*) FROM read_csv_auto('{raw_path}') GROUP BY 1 ORDER BY 2 DESC").fetchall():
    print(row)

## Шаг 3. Пять проверок качества

In [ ]:
print("1. unique_id (дубликаты):", con.execute(f"SELECT COUNT(*) FROM (SELECT id FROM read_csv_auto('{raw_path}') GROUP BY id HAVING COUNT(*) > 1)").fetchone()[0])
print("2. valid_room_type (неверный тип):", con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{raw_path}') WHERE room_type NOT IN ('Entire home/apt', 'Private room', 'Shared room', 'Hotel room')").fetchone()[0])
print("3. valid_availability (дней вне 0-365):", con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{raw_path}') WHERE availability_365 < 0 OR availability_365 > 365").fetchone()[0])
print("4. not_null_price (пропуски цены):", con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{raw_path}') WHERE price IS NULL").fetchone()[0])
print("5. positive_price (отрицательные цены):", con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{raw_path}') WHERE price < 0").fetchone()[0])

## Шаг 4. Запись staging и карантина (с сохранением NULL)

In [ ]:
con.execute(f"COPY (SELECT *, 'neg_price' as rule_id, 'Negative price' as reason FROM read_csv_auto('{raw_path}') WHERE price < 0) TO '{quarantine_path}' (FORMAT PARQUET);")
con.execute(f"COPY (SELECT *, CASE WHEN price IS NULL THEN true ELSE false END as flag_null_price, CASE WHEN availability_365 < 0 OR availability_365 > 365 THEN true ELSE false END as flag_invalid_availability FROM read_csv_auto('{raw_path}') WHERE price >= 0 OR price IS NULL) TO '{staging_path}' (FORMAT PARQUET);")
print("Сохранено")

## Шаг 5. Проверка баланса строк

In [ ]:
raw_c = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{raw_path}')").fetchone()[0]
stg_c = con.execute(f"SELECT COUNT(*) FROM read_parquet('{staging_path}')").fetchone()[0]
qtn_c = con.execute(f"SELECT COUNT(*) FROM read_parquet('{quarantine_path}')").fetchone()[0]
print(f"Raw: {raw_c}, Staging: {stg_c}, Quarantine: {qtn_c}")
print(f"Баланс сходится: {raw_c == stg_c + qtn_c}")